# Fraud Detection — Machine Learning Model

This notebook:
1. Loads the enriched transactions parquet
2. Assembles the feature vector using PySpark MLlib
3. Trains a Random Forest Classifier (100 trees)
4. Evaluates: AUC, precision, recall, F1-score
5. Prints feature importances
6. Saves the model and metrics

In [1]:
# Se importan os y json para manejar rutas y archivos de configuración/métricas
import os
import json

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Carpeta donde están los datos procesados (parquet enriquecido del batch processing)
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
# Carpeta curated donde se guardarán el modelo entrenado y las métricas
CURATED_DIR   = os.path.join(BASE_DIR, "curated")
# Ruta al parquet de entrada que contiene todas las features calculadas
ENRICHED_PATH = os.path.join(PROCESSED_DIR, "transactions_enriched")
# Ruta donde se guardará el modelo de Random Forest entrenado
MODEL_PATH    = os.path.join(CURATED_DIR, "fraud_model")
# Ruta donde se guardará el JSON con las métricas de evaluación del modelo
METRICS_PATH  = os.path.join(CURATED_DIR, "model_metrics.json")

# Se crea la carpeta curated si no existe para poder guardar el modelo y las métricas
os.makedirs(CURATED_DIR, exist_ok=True)

print(f"Enriched parquet: {ENRICHED_PATH}")
print(f"Model output    : {MODEL_PATH}")
print(f"Metrics output  : {METRICS_PATH}")

Enriched parquet: d:\UP\BigData\Pryecto1\processed\transactions_enriched
Model output    : d:\UP\BigData\Pryecto1\curated\fraud_model
Metrics output  : d:\UP\BigData\Pryecto1\curated\model_metrics.json


## 1. Start Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

# Use project directory for Spark temp — avoids Windows cleaning AppData\Local\Temp mid-session
SPARK_TEMP = os.path.abspath(os.path.join(os.getcwd(), "..", "spark_tmp"))
os.makedirs(SPARK_TEMP, exist_ok=True)

print("Starting Spark session ...")

spark = (
    SparkSession.builder
    .appName("FraudDetection_MLModel")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .config("spark.local.dir", SPARK_TEMP)
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.maxResultSize", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✓ Spark session started — version {spark.version}")
print(f"✓ Spark temp dir: {SPARK_TEMP}")

Starting Spark session ...
✓ Spark session started — version 4.1.1
✓ Spark temp dir: d:\UP\BigData\Pryecto1\spark_tmp


## 2. Load Enriched Transactions

In [3]:
import pandas as pd
from pyspark.sql.functions import col, trim

print("=" * 60)
print("STEP 2 — Loading enriched transactions parquet + fraud labels")
print("=" * 60)

df = spark.read.parquet(ENRICHED_PATH)
df = df.drop("is_fraud")
df.printSchema()
print(f"  ✓ Loaded {df.count():,} rows")

LABELS_PATH = os.path.join(BASE_DIR, "raw", "train_fraud_labels.json")

with open(LABELS_PATH, encoding="utf-8") as f:
    labels_raw = json.load(f)

labels_dict = labels_raw["target"]
labels_pd = pd.DataFrame({
    "id":       list(labels_dict.keys()),
    "is_fraud": [1 if v == "Yes" else 0 for v in labels_dict.values()]
})
print(f"  ✓ Labels loaded ({len(labels_pd):,} rows)")

# Write to CSV so Spark reads it natively — avoids Python worker OOM crash
LABELS_TEMP = os.path.join(PROCESSED_DIR, "labels_temp.csv")
labels_pd.to_csv(LABELS_TEMP, index=False)

labels_df = (spark.read.csv(LABELS_TEMP, header=True, inferSchema=True)
    .withColumn("id", trim(col("id").cast("string")))
    .withColumn("is_fraud", col("is_fraud").cast("integer")))

df = df.withColumn("id", trim(col("id").cast("string")))
df = df.join(labels_df, on="id", how="left")

print("Label distribution:")
df.groupBy("is_fraud").count().orderBy("is_fraud").show()


STEP 2 — Loading enriched transactions parquet + fraud labels
root
 |-- id: string (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- amount: float (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: string (nullable = true)
 |-- errors: string (nullable = true)
 |-- card_client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: string (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: integ

## 3. Prepare Features

In [4]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col, when, isnan

print("=" * 60)
print("STEP 3 — Preparing feature vector")
print("=" * 60)

FEATURE_COLS = [
    "amount_deviation",
    "transaction_velocity",
    "card_utilization",
    "mcc_fraud_rate",
    "is_weekend",
    "is_night",
    "amount",
]

available = [c for c in FEATURE_COLS if c in df.columns]
missing   = [c for c in FEATURE_COLS if c not in df.columns]

print(f"  Available features : {available}")
if missing:
    print(f"  Missing features   : {missing}  (will be filled with 0)")
    for c in missing:
        df = df.withColumn(c, F.lit(0.0))

label_col = "is_fraud"

# Cast to double and replace both null AND float NaN with 0.0
# Note: fillna() does NOT replace float NaN values, only null —
# VectorAssembler with handleInvalid="skip" would silently drop every row
for c in FEATURE_COLS:
    df = df.withColumn(c, col(c).cast("double"))
    df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), 0.0).otherwise(col(c)))

# Convert boolean features to integer
for c in ["is_weekend", "is_night"]:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast("integer"))

# Diagnose label-join quality before filtering
total_rows  = df.count()
labeled_rows = df.filter(col(label_col).isNotNull()).count()
print(f"  Total rows           : {total_rows:,}")
print(f"  Rows with label      : {labeled_rows:,}")
print(f"  Rows without label   : {total_rows - labeled_rows:,}")

if labeled_rows == 0:
    # Show a sample of IDs from both sides to help diagnose the mismatch
    print("Sample parquet IDs:")
    df.select("id").show(5, truncate=False)
    raise ValueError(
        "No rows have a non-null is_fraud label after the join in Step 2. "
        "The 'id' format in the parquet likely does not match the keys in "
        "train_fraud_labels.json. Check the sample IDs printed above vs the JSON keys."
    )

# Cast label to integer, drop rows where label is null
df = df.withColumn(label_col, col(label_col).cast("integer"))
df_ml = df.select(FEATURE_COLS + [label_col]).filter(col(label_col).isNotNull())

print(f"  Rows after filtering nulls: {df_ml.count():,}")
print("  Label distribution:")
df_ml.groupBy(label_col).count().orderBy(label_col).show()

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="features",
    handleInvalid="error"  # safe now — nulls and NaNs already cleaned above
)
df_vec = assembler.transform(df_ml)
print(f"  ✓ Feature vector assembled — column 'features' created")
df_vec.select("features", label_col).show(3, truncate=False)


STEP 3 — Preparing feature vector
  Available features : ['amount_deviation', 'transaction_velocity', 'card_utilization', 'mcc_fraud_rate', 'is_weekend', 'is_night', 'amount']
  Total rows           : 13,305,915
  Rows with label      : 8,914,963
  Rows without label   : 4,390,952
  Rows after filtering nulls: 8,914,963
  Label distribution:
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       0|8901631|
|       1|  13332|
+--------+-------+

  ✓ Feature vector assembled — column 'features' created
+--------------------------------------------------------+--------+
|features                                                |is_fraud|
+--------------------------------------------------------+--------+
|(7,[0,1,6],[-0.3719528995335819,2.0,12.119999885559082])|0       |
|(7,[0,1,6],[1.5512827710205312,1.0,131.22000122070312]) |0       |
|(7,[0,1,6],[-0.36119857454674065,2.0,1.059999942779541])|0       |
+--------------------------------------------------------+--------+
only sho

## 4. Train/Test Split

In [5]:
print("=== Columns in df ===")
print(df.columns)

print("\n=== Null counts per feature column ===")
from pyspark.sql.functions import col, sum as spark_sum
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) 
           for c in FEATURE_COLS + ["is_fraud"] if c in df.columns]).show()

print("\n=== Sample rows (raw) ===")
df.select([c for c in FEATURE_COLS if c in df.columns]).show(5, truncate=False)

=== Columns in df ===
['id', 'card_id', 'date', 'client_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'card_client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'mcc_code', 'description', 'transaction_velocity', 'amount_deviation', 'is_weekend', 'is_night', 'mcc_fraud_rate', 'card_utilization', 'is_fraud']

=== Null counts per feature column ===
+----------------+--------------------+----------------+--------------+----------+--------+------+--------+
|amount_deviation|transaction_velocity|card_utilization|mcc_fraud_rate|is_weekend|is_night|amount|is_fraud|
+----------------+--------------------+----------------

In [6]:
print("=" * 60)
print("STEP 4 — Train / Test split (80 / 20, seed=42)")
print("=" * 60)

total_vec = df_vec.count()
print(f"  Rows entering split: {total_vec:,}")

if total_vec == 0:
    raise ValueError("df_vec is empty — check Step 3: dropna() may have removed all rows. Run df_ml.count() and df.select(FEATURE_COLS).show() to inspect nulls.")

train_df, test_df = df_vec.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count  = test_df.count()

print(f"  Training rows : {train_count:,}")
print(f"  Test rows     : {test_count:,}")
print(f"  Split ratio   : {train_count / (train_count + test_count):.1%} / {test_count / (train_count + test_count):.1%}")

print("=== Columns in df ===")
print(df.columns)

print("\n=== Null counts per feature column ===")
from pyspark.sql.functions import col, sum as spark_sum
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) 
           for c in FEATURE_COLS + ["is_fraud"] if c in df.columns]).show()

print("\n=== Sample rows (raw) ===")
df.select([c for c in FEATURE_COLS if c in df.columns]).show(5, truncate=False)



STEP 4 — Train / Test split (80 / 20, seed=42)
  Rows entering split: 8,914,963
  Training rows : 7,132,522
  Test rows     : 1,782,441
  Split ratio   : 80.0% / 20.0%
=== Columns in df ===
['id', 'card_id', 'date', 'client_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'card_client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'mcc_code', 'description', 'transaction_velocity', 'amount_deviation', 'is_weekend', 'is_night', 'mcc_fraud_rate', 'card_utilization', 'is_fraud']

=== Null counts per feature column ===
+----------------+--------------------+----------------+--------------+----------+--------+------+-----

## 5. Train Random Forest Classifier

In [7]:
from pyspark.ml.classification import RandomForestClassifier

print("=" * 60)
print("STEP 5 — Training Random Forest (100 trees)")
print("=" * 60)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    numTrees=100,
    seed=42,
    maxDepth=10,
    minInstancesPerNode=5
)

print("  Fitting model on training data ...")
import time
t0 = time.time()
rf_model = rf.fit(train_df)
elapsed = time.time() - t0

print(f"  ✓ Model trained in {elapsed:.1f}s")
print(f"  Number of trees : {rf_model.getNumTrees}")
print(f"  Total nodes     : {rf_model.totalNumNodes}")

STEP 5 — Training Random Forest (100 trees)


  Fitting model on training data ...
  ✓ Model trained in 1204.2s
  Number of trees : 100
  Total nodes     : 226


## 6. Evaluate the Model

In [8]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print("=" * 60)
print("STEP 6 — Evaluating model on test set")
print("=" * 60)

predictions = rf_model.transform(test_df).cache()
predictions.count()  # materialize cache before evaluation
print(f"  ✓ Predictions cached ({predictions.count():,} rows)")

# ── AUC (Binary) — sampled to avoid OOM on sort ───────────────────────────────
# 20% is statistically sufficient for AUC with millions of rows
auc_sample = predictions.sample(fraction=0.2, seed=42).cache()
auc_sample.count()

binary_eval = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = binary_eval.evaluate(auc_sample)
print(f"  AUC (area under ROC)   : {auc:.4f}  (estimated on 20% sample)")

# ── Multiclass metrics — run on full set ─────────────────────────────────────
mc_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction"
)

accuracy  = mc_eval.evaluate(predictions, {mc_eval.metricName: "accuracy"})
precision = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedPrecision"})
recall    = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedRecall"})
f1        = mc_eval.evaluate(predictions, {mc_eval.metricName: "f1"})

print(f"  Accuracy               : {accuracy:.4f}")
print(f"  Weighted Precision     : {precision:.4f}")
print(f"  Weighted Recall        : {recall:.4f}")
print(f"  Weighted F1-Score      : {f1:.4f}")

# Show sample predictions
print("\n  Sample predictions (actual vs predicted):")
predictions.select(label_col, "prediction", "probability").show(10, truncate=False)

# Confusion matrix summary
print("  Prediction distribution:")
predictions.groupBy(label_col, "prediction").count().orderBy(label_col, "prediction").show()

auc_sample.unpersist()
predictions.unpersist()


STEP 6 — Evaluating model on test set
  ✓ Predictions cached (1,782,826 rows)
  AUC (area under ROC)   : 0.7167  (estimated on 20% sample)
  Accuracy               : 0.9985
  Weighted Precision     : 0.9971
  Weighted Recall        : 0.9985
  Weighted F1-Score      : 0.9978

  Sample predictions (actual vs predicted):
+--------+----------+------------------------------------------+
|is_fraud|prediction|probability                               |
+--------+----------+------------------------------------------+
|0       |0.0       |[0.9984890277348716,0.0015109722651284465]|
|0       |0.0       |[0.9985378371775395,0.0014621628224604673]|
|0       |0.0       |[0.998523905375302,0.001476094624698066]  |
|0       |0.0       |[0.9984890277348716,0.0015109722651284465]|
|0       |0.0       |[0.9985215850562319,0.0014784149437680968]|
|0       |0.0       |[0.9984890277348716,0.0015109722651284465]|
|0       |0.0       |[0.9984890277348716,0.0015109722651284465]|
|0       |0.0       |[0.998529

DataFrame[amount_deviation: double, transaction_velocity: double, card_utilization: double, mcc_fraud_rate: double, is_weekend: int, is_night: int, amount: double, is_fraud: int, features: vector, rawPrediction: vector, probability: vector, prediction: double]

## 7. Feature Importances

In [9]:
print("=" * 60)
print("STEP 7 — Feature importances (ranked)")
print("=" * 60)

importances = rf_model.featureImportances.toArray()

feat_imp = sorted(
    zip(FEATURE_COLS, importances),
    key=lambda x: x[1],
    reverse=True
)

print(f"  {'Feature':<30} {'Importance':>12}")
print(f"  {'-'*30} {'-'*12}")
for feat, imp in feat_imp:
    bar = "█" * int(imp * 50)
    print(f"  {feat:<30} {imp:>12.4f}  {bar}")

STEP 7 — Feature importances (ranked)
  Feature                          Importance
  ------------------------------ ------------
  amount                               0.4191  ████████████████████
  amount_deviation                     0.3299  ████████████████
  transaction_velocity                 0.2163  ██████████
  is_night                             0.0286  █
  is_weekend                           0.0062  
  card_utilization                     0.0000  
  mcc_fraud_rate                       0.0000  


## 8 — Save Model and Metrics

In [10]:
print("=" * 60)
print("STEP 8 — Saving model and metrics")
print("=" * 60)

# Save model
rf_model.write().overwrite().save(MODEL_PATH)
print(f"  ✓ Model saved to {MODEL_PATH}")

# Save metrics JSON
metrics = {
    "model":           "RandomForestClassifier",
    "num_trees":       100,
    "features":        FEATURE_COLS,
    "train_rows":      train_count,
    "test_rows":       test_count,
    "auc":             round(float(auc), 4),
    "accuracy":        round(float(accuracy), 4),
    "precision":       round(float(precision), 4),
    "recall":          round(float(recall), 4),
    "f1_score":        round(float(f1), 4),
    "feature_importances": {
        feat: round(float(imp), 6) for feat, imp in feat_imp
    }
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"  ✓ Metrics saved to {METRICS_PATH}")
print(f"\n  Summary:")
print(json.dumps(
    {k: v for k, v in metrics.items() if k != "feature_importances"},
    indent=4
))

STEP 8 — Saving model and metrics
  ✓ Model saved to d:\UP\BigData\Pryecto1\curated\fraud_model
  ✓ Metrics saved to d:\UP\BigData\Pryecto1\curated\model_metrics.json

  Summary:
{
    "model": "RandomForestClassifier",
    "num_trees": 100,
    "features": [
        "amount_deviation",
        "transaction_velocity",
        "card_utilization",
        "mcc_fraud_rate",
        "is_weekend",
        "is_night",
        "amount"
    ],
    "train_rows": 7132522,
    "test_rows": 1782441,
    "auc": 0.7167,
    "accuracy": 0.9985,
    "precision": 0.9971,
    "recall": 0.9985,
    "f1_score": 0.9978
}


In [11]:
print("Stopping Spark session ...")
spark.stop()
print("✓ Spark session stopped. ML pipeline complete.")

Stopping Spark session ...
✓ Spark session stopped. ML pipeline complete.
